In [29]:
import os

# Fallback to classic HTTP downloads (Xet can be blocked on some networks)
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [30]:
import huggingface_hub
from huggingface_hub import constants

print(huggingface_hub.__version__)
print(constants.HF_HUB_DISABLE_XET)

0.36.2
True


In [31]:
HYPERPARAMS = {
    "model_id": "microsoft/Phi-3-mini-4k-instruct",
    # LoRA: which part of the model we retrain
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    # Training
    "num_train_epochs": 10,
    "learning_rate": 3e-4,
    "per_device_train_batch_size": 16,
    "max_seq_length": 64,
    "validation_size": 0.1,
    "seed": 42,
}

In [32]:
from peft import LoraConfig
from trl import SFTConfig

lora_config = LoraConfig(
    r=HYPERPARAMS["lora_r"],
    lora_alpha=HYPERPARAMS["lora_alpha"],
    lora_dropout=HYPERPARAMS["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="./phi3-yoda-adapter",
    num_train_epochs=HYPERPARAMS["num_train_epochs"],
    learning_rate=HYPERPARAMS["learning_rate"],
    per_device_train_batch_size=HYPERPARAMS["per_device_train_batch_size"],
    max_seq_length=HYPERPARAMS["max_seq_length"],
    packing=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",  # "paged_adamw_8bit" in the cloud, needs bitsandbytes
    logging_steps=10,
    eval_strategy="epoch",
    report_to=["tensorboard"],
    logging_dir="./logs",
    seed=HYPERPARAMS["seed"],
)

print(lora_config)

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type='CAUSAL_LM', inference_mode=False, r=8, target_modules={'o_proj', 'gate_up_proj', 'qkv_proj', 'down_proj'}, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


In [33]:
from accelerate import init_empty_weights
from peft import get_peft_model
from transformers import AutoConfig, AutoModelForCausalLM

model_config = AutoConfig.from_pretrained(HYPERPARAMS["model_id"])
with init_empty_weights():
    model = AutoModelForCausalLM.from_config(model_config)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


In [34]:
print(peft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (embed_dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3SdpaAttention(
              (o_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_ve

In [35]:
import json

import gcsfs
import pandas as pd
from datasets import Dataset

TRAIN_URI = "gs://clem-andrian-llmops/pipeline_root/123905721265/yoda-model-training-pipeline-20261022193659/data-transformation-component_6557552356681056256/train_dataset_path"

fs = gcsfs.GCSFileSystem()
with fs.open(TRAIN_URI) as f:
    df = pd.read_csv(f)

conversations = [json.loads(m) for m in df["messages"]]
dataset = Dataset.from_dict({"messages": conversations})

splits = dataset.train_test_split(
    test_size=HYPERPARAMS["validation_size"], seed=HYPERPARAMS["seed"]
)
train_dataset, eval_dataset = splits["train"], splits["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['messages'],
    num_rows: 518
})
Dataset({
    features: ['messages'],
    num_rows: 58
})


In [36]:
import json

import pandas as pd
from datasets import Dataset

TRAIN_URI = "gs://clem-andrian-llmops/pipeline_root/123905721265/yoda-model-training-pipeline-20261022193659/data-transformation-component_6557552356681056256/train_dataset_path"  # paste yours

import gcsfs

fs = gcsfs.GCSFileSystem()
with fs.open(TRAIN_URI) as f:
    df = pd.read_csv(f)# Each conversation was saved as JSON text in session 2, so we convert it back
conversations = [json.loads(m) for m in df["messages"]]
dataset = Dataset.from_dict({"messages": conversations})

splits = dataset.train_test_split(
    test_size=HYPERPARAMS["validation_size"], seed=HYPERPARAMS["seed"]
)
train_dataset, eval_dataset = splits["train"], splits["test"]

print(train_dataset)
print(eval_dataset)
print(train_dataset[0]["messages"])

Dataset({
    features: ['messages'],
    num_rows: 518
})
Dataset({
    features: ['messages'],
    num_rows: 58
})
[{'role': 'user', 'content': 'Eight miles of woodland burned to waste.'}, {'role': 'assistant', 'content': 'Burned to waste, eight miles of woodland did.'}]


In [37]:
import fsspec
import pandas

print("pandas:", pandas.__version__)
print("fsspec:", fsspec.__version__)

pandas: 2.2.3
fsspec: 2025.3.0


In [38]:
from transformers import AutoTokenizer, Phi3Config

tokenizer = AutoTokenizer.from_pretrained(HYPERPARAMS["model_id"])
# Same fix as in the article: the EOS token must not be used for padding
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

tiny_config = Phi3Config(
    vocab_size=32064,
    hidden_size=64,
    intermediate_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    pad_token_id=tokenizer.pad_token_id,
)
tiny_model = AutoModelForCausalLM.from_config(tiny_config)

print(tokenizer.apply_chat_template(train_dataset[0]["messages"], tokenize=False))

<|user|>
Eight miles of woodland burned to waste.<|end|>
<|assistant|>
Burned to waste, eight miles of woodland did.<|end|>
<|endoftext|>


In [39]:
from trl import SFTTrainer

local_test_config = SFTConfig(
    output_dir="./local-test-run",
    max_steps=5,
    per_device_train_batch_size=4,
    max_seq_length=HYPERPARAMS["max_seq_length"],
    packing=True,
    learning_rate=HYPERPARAMS["learning_rate"],
    optim="adamw_torch",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=5,
    report_to=["tensorboard"],
    logging_dir="./logs/local-test",
)

trainer = SFTTrainer(
    model=tiny_model,
    args=local_test_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

/Users/andrianrajesy/Documents/Albert School/MSC2/ Agentic workflows/.venv/lib/python3.11/site-packages/peft/mapping.py:172: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'microsoft/Phi-3-mini-4k-instruct' to ''. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/Users/andrianrajesy/Documents/Albert School/MSC2/ Agentic workflows/.venv/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:403: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
/Users/andrianrajesy/Documents/Albert School/MSC2/ Agentic workflows/.venv/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:428: User

{'loss': 10.2153, 'grad_norm': 0.2833576798439026, 'learning_rate': 0.00023999999999999998, 'epoch': 0.02}
{'loss': 10.2103, 'grad_norm': 0.3172939717769623, 'learning_rate': 0.00017999999999999998, 'epoch': 0.03}
{'loss': 10.204, 'grad_norm': 0.30181246995925903, 'learning_rate': 0.00011999999999999999, 'epoch': 0.05}
{'loss': 10.2049, 'grad_norm': 0.25045594573020935, 'learning_rate': 5.9999999999999995e-05, 'epoch': 0.06}
{'loss': 10.1975, 'grad_norm': 0.2661823332309723, 'learning_rate': 0.0, 'epoch': 0.08}



100%|██████████| 5/5 [00:00<00:00,  8.73it/s]

{'eval_loss': 10.37667179107666, 'eval_runtime': 0.0811, 'eval_samples_per_second': 357.782, 'eval_steps_per_second': 49.349, 'epoch': 0.08}


/Users/andrianrajesy/Documents/Albert School/MSC2/ Agentic workflows/.venv/lib/python3.11/site-packages/peft/utils/save_and_load.py:243: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(
100%|██████████| 5/5 [00:15<00:00,  3.08s/it]

{'train_runtime': 15.4006, 'train_samples_per_second': 1.299, 'train_steps_per_second': 0.325, 'train_loss': 10.20639533996582, 'epoch': 0.08}


TrainOutput(global_step=5, training_loss=10.20639533996582, metrics={'train_runtime': 15.4006, 'train_samples_per_second': 1.299, 'train_steps_per_second': 0.325, 'total_flos': 16501800960.0, 'train_loss': 10.20639533996582, 'epoch': 0.07936507936507936})

In [40]:
from pathlib import Path

from google.cloud import storage


def download_model(model_uri: str, local_dir: str) -> None:
    """Download model from GCS to local directory."""
    bucket_name, _, prefix = model_uri.removeprefix("gs://").partition("/")
    prefix = prefix.rstrip("/")
    client = storage.Client()
    for blob in client.list_blobs(bucket_name, prefix=prefix + "/"):
        if blob.name.endswith("/"):
            continue
        local_path = Path(local_dir) / Path(blob.name).relative_to(prefix)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(local_path)

In [41]:
def build_prompt(tokenizer: AutoTokenizer, sentence: str) -> str:
    """Build a prompt from a sentence applying the chat template."""
    messages = [{"role": "user", "content": sentence}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


print(build_prompt(tokenizer, "The Force is strong in you!"))

<|user|>
The Force is strong in you!<|end|>
<|assistant|>



In [42]:
from typing import Any

import torch


def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    **kwargs: Any,
) -> str:
    """Generate a response from the model given a prompt."""
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(**inputs, eos_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.batch_decode(output_ids, skip_special_tokens=False)[0]


raw_output = generate_response(
    trainer.model, tokenizer, build_prompt(tokenizer, "The Force is strong in you!"),
    max_new_tokens=10,
)
print(raw_output)

<|user|> The Force is strong in you!<|end|><|assistant|>евич Le drop которая rankszigotic� inspirстори


In [43]:
import re


def extract_response(model_output: str) -> str:
    """Extract the actual response from the model output."""
    match = re.search(
        r"<\|assistant\|>(.*?)(?:<\|end\|>|<\|endoftext\|>|$)", model_output, re.DOTALL
    )
    return match.group(1).strip() if match else model_output.strip()


example = """<|user|>
The Force is strong in you!<|end|>
<|assistant|>
Strong in you, the Force is. Yes, hrrmmm.<|end|>"""
print(extract_response(example))

Strong in you, the Force is. Yes, hrrmmm.


In [44]:
print(extract_response(raw_output))

евич Le drop которая rankszigotic� inspirстори


In [45]:
TEST_URI = TRAIN_URI.replace("train_dataset_path", "test_dataset_path")

with fs.open(TEST_URI) as f:
    test_df = pd.read_csv(f)

rows = []
for messages_json in test_df["messages"].head(3):
    messages = json.loads(messages_json)
    sentence = messages[0]["content"]   # user message
    reference = messages[1]["content"]  # expected Yoda answer

    prompt = build_prompt(tokenizer, sentence)
    raw_output = generate_response(trainer.model, tokenizer, prompt, max_new_tokens=64)
    rows.append({
        "user_input": sentence,
        "reference": reference,
        "extracted_response": extract_response(raw_output),
    })

predictions_df = pd. DataFrame(rows)
predictions_df.to_csv("predictions_local_test.csv", index=False)
predictions_df

,user_input,reference,extracted_response
0,He wrote down a long list of items.,"A long list of items, down he wrote. Hmm.",linketonCur利 Keep жовтняsyncщо Ke boat Monsieu...
1,The duke left the park in a silver coach.,"Left the park in a silver coach, the duke did.","choosebganka headrii navigation simult“,学цяϕђ ..."
2,A siege will crack the strong defense.,"Crack the strong defense, a siege will.",link Украї Посилання unders janvier Kilometer ...


In [46]:
from ragas.metrics import BleuScore, ExactMatch, NonLLMStringSimilarity, RougeScore

predictions_df = pd.read_csv("predictions_local_test.csv")

metric_definitions = [
    RougeScore(name="rouge_l", rouge_type="rougeL"),
    BleuScore(),
    NonLLMStringSimilarity(),
    ExactMatch(),
]
print([m.name for m in metric_definitions])

['rouge_l', 'bleu_score', 'non_llm_string_similarity', 'exact_match']


In [48]:
from ragas import SingleTurnSample
from ragas.metrics.base import SingleTurnMetric


def compute_metrics(
    user_input: str,
    response: str,
    reference: str,
    metric_definitions: list[SingleTurnMetric],
) -> dict[str, float | int]:
    """Compute metrics for a single prediction."""
    sample = SingleTurnSample(user_input=user_input, response=response, reference=reference)
    return {metric.name: metric.single_turn_score(sample) for metric in metric_definitions}


# Sanity checks: a perfect answer should score 1, a random one close to 0
reference = "Crack the strong defense, a siege will."
print(compute_metrics("A siege will crack the strong defense.", reference, reference, metric_definitions))
print(compute_metrics("A siege will crack the strong defense.", "onyme face estimate", reference, metric_definitions))

{'rouge_l': 1.0, 'bleu_score': 1.0000000000000004, 'non_llm_string_similarity': 1.0, 'exact_match': 1.0}
{'rouge_l': 0.0, 'bleu_score': 0.0, 'non_llm_string_similarity': 0.17948717948717952, 'exact_match': 0.0}


In [49]:
per_sample_metrics = pd.DataFrame([
    {
        **row,
        **compute_metrics(
            row["user_input"],
            str(row["extracted_response"]),
            row["reference"],
            metric_definitions,
        ),
    }
    for row in predictions_df.to_dict("records")
])
per_sample_metrics

,user_input,reference,extracted_response,rouge_l,bleu_score,non_llm_string_similarity,exact_match
0,He wrote down a long list of items.,"A long list of items, down he wrote. Hmm.",linketonCur利 Keep жовтняsyncщо Ke boat Monsieu...,0.0,0.000000,0.082803,0.0
1,The duke left the park in a silver coach.,"Left the park in a silver coach, the duke did.","choosebganka headrii navigation simult“,学цяϕђ ...",0.0,0.009451,0.116838,0.0
2,A siege will crack the strong defense.,"Crack the strong defense, a siege will.",link Украї Посилання unders janvier Kilometer ...,0.0,0.000000,0.073048,0.0


In [50]:
def aggregate_metrics(per_sample_metrics: pd.DataFrame) -> dict[str, float]:
    """Aggregate metrics from per-sample metrics DataFrame."""
    metric_columns = [m.name for m in metric_definitions]
    return {f"{name}_mean": float(per_sample_metrics[name].mean()) for name in metric_columns}


aggregate_metrics(per_sample_metrics)

{'rouge_l_mean': 0.0,
 'bleu_score_mean': 0.0031504194592730152,
 'non_llm_string_similarity_mean': 0.09089629822842489,
 'exact_match_mean': 0.0}